In [ ]:
import sys
from pathlib import Path

BASE_DIR = Path.cwd().parent
sys.path.append(str(BASE_DIR))

In [ ]:
import json
import os
from datetime import datetime

import pandas as pd
from tqdm import tqdm

from maintenance_llm.llm_client.ollama_client import ollama_client
from maintenance_llm.prompts.prompt_builder import build_prompt
from maintenance_llm.rag_module.bge import bge_retriever
from maintenance_llm.rag_module.context_compiler import compile_context
from maintenance_llm.rag_module.minilm import minilm_retriever
from maintenance_llm.rag_module.tfidf import tfidf_retriever
from maintenance_llm.utils.paths import LABEL_DATABASE_FILE, OUTPUT_DIR

endpoint_url = os.getenv("OLLAMA_ENDPOINT_URL")

Initialise Model List and Notebook-based Functions

In [ ]:
"""define the list of models for analysis"""

model_library = ["mistral:7b", "qwen3.5:9b", "llama3:8b", "gemma3:4b"]

In [ ]:
"""Function to initialise the selected retriever based on the provided name and context database"""

def get_retriever(retriever_name: str, context_db: list[str], record_retrieval: bool = False):
    retrievers = {
        "tfidf": tfidf_retriever,
        "minilm": minilm_retriever,
        "bge": bge_retriever,
        
    }

    if retriever_name not in retrievers:
        raise ValueError(
            f"Unsupported retriever: {retriever_name}"
        )

    return retrievers[retriever_name](context_db,record_retrieval)

In [ ]:
"""Main Function - run the selected model on the provided dataset and return the output as a pd.DataFrame"""

def run_ollama(model_name:str, dataset: pd.DataFrame, retriever_model: str, RAG: bool = False, json_format: bool=False) -> pd.DataFrame:
    # Initiate run
    print(f"Run initiated for model: {model_name}")
    print(f"Analysis: Open-source Model | RAG: {RAG} | JSON format: {json_format}")
    print(f"initiating classification for {len(dataset['ServiceOrderText'].tolist())} sampled texts... \n")
    
    # Prepare input texts and valid labels for classification
    input_texts = dataset['ServiceOrderText'].tolist()
    valid_labels = dataset['Failure Category'].unique().tolist()
    
    # Initiate Ollama Client with the selected model
    Ollama_client = ollama_client(model = model_name, valid_labels = valid_labels) 

    if RAG: 
        # Load context database from json 
        with open(LABEL_DATABASE_FILE, "r", encoding= "utf-8") as f: 
            context_data = json.load(f)
                        
        context_db = compile_context(context_data) # compile context data into formtted list
        retriever = get_retriever(retriever_model, context_db)  # initialise selected retriever
        retrieved_df = retriever.retrieve_context(input_texts)  # returns df of input_texts and corresponding top k context data 
        
        inference_df = retrieved_df
    
    else: 
        inference_df = pd.DataFrame({"input_text": input_texts})

    
    full_output = [] 
    # LLM inference loop
    for _, row in tqdm(inference_df.iterrows(),total=len(inference_df), desc="Running inference"):
        text = row["input_text"]
        
        if RAG: 
            prompt = build_prompt(rag= RAG, text=text, retrieved_context= row["retrieved_context"], labels=valid_labels)
        else: 
            prompt = build_prompt(rag= RAG, text=text, labels=valid_labels)
        
        # LLM inference
        LLM_output = Ollama_client.predict(prompt,json_format)

        full_output.append(LLM_output) 
    
    # Compile final output into pd dataframe with SO text and true label
    output_df = pd.DataFrame(full_output)
    output_df.insert(0, "ServiceOrderText", dataset["ServiceOrderText"].tolist())
    output_df.insert(1, "True_label", dataset["Failure Category"].tolist())
    
    print("Run completed. \n")

    return output_df


In [ ]:
"""Function to save the results into csv file"""

def save_result(model_name:str, result_df: pd.DataFrame, retriever_model: str, RAG: bool = False, json_format=False):
    timestamp = datetime.now().strftime("%d-%m-%Y_%HH%MM") #timestamp for record
      
    # configure file name and save location 
    model_name = model_name.replace(":", "-").replace("/", "-")
    output_dir = OUTPUT_DIR / "open-source_models"
    output_dir.mkdir(parents=True, exist_ok=True)
    
    if RAG: 
        prompt_name = "RAG"
    else: 
        prompt_name = "base"  

    if json_format:
        result_df.to_csv(output_dir / f"json_{prompt_name}_{retriever_model}_{model_name}_{timestamp}.csv", index=False) 
    else:
        result_df.to_csv(output_dir / f"{prompt_name}_{retriever_model}_{model_name}_{timestamp}.csv", index=False)
    
    print (f"Results saved for model: {model_name} using prompt version: {prompt_name} \n\n")

Sample Run with [Model: Qwen3.5-9B] 

In [ ]:
# load dataset
dataset = pd.read_csv(f"{BASE_DIR}/results_public/data_example/dataset_example.csv")
dataset.shape

In [ ]:
# Run parameter configurations
run_library = model_library  # edit to execute specific models from the model_library list
rag_toggle = True
json_toggle = True
retriever = "tfidf"

In [ ]:
# run inference loop 
for model in run_library:
    result = run_ollama(model_name = model, dataset = dataset, RAG=rag_toggle, retriever_model=retriever, json_format=json_toggle)
    save_result(model_name = model, result_df = result, RAG=rag_toggle, retriever_model=retriever, json_format=json_toggle)